# 爆炸超压拟合公式优化 — Z 分段独立拟合

## 策略
诊断结论：Z=4（远场边界）的误差模式与 Z=1/2/3 有本质差异——
Z=4 处的前向 θ=0° 严重欠预测（原 max=58%），说明该距离段的物理机制（速度-角度耦合）
与中近场不同，单一参数组无法兼顾。

**方案**：将数据按 Z 分为两段，各自独立拟合 33 参数模型：
- **段A (Z=1, 2, 3)**：2940 样本，物理上属中近场，原拟合已较好
- **段B (Z=4)**：980 样本，远场边界，需独立的系数捕捉其特殊的速度-角度耦合

分段边界设在 Z=3.5，覆盖所有整数 Z 值。两段模型结构完全相同（均为 33 参数），
仅系数不同，使用时按 Z 查表选择。

In [ ]:
%% 第一步：读取数据与静爆基准公式拟合 (15参数)
disp('>>> 步骤1：读取数据与静爆15参数拟合...');

% 1. 读取数据
opts = detectImportOptions('zhuxing1.xlsx');
opts.VariableNamingRule = 'preserve';
df = readtable('zhuxing1.xlsx', opts);
df.Properties.VariableNames(1:6) = {'M', 'H', 'V', 'P_MPa', 'Z', 'theta'};

% 2. 预处理环境因子 (萨克斯定律)
T0 = 288.15;
df.T_h = T0 - 6.5 * df.H;
df.Sp = (df.T_h / T0) .^ 5.25588;

% 3. 提取静爆数据并拟合
df_static = df(df.V == 0, :);

static_model = @(p, X) ...
    (X(:,2).^(2/3) ./ X(:,1))   .* ( p(1) + p(2).*cosd(X(:,3)) + p(3).*cosd(2*X(:,3)) + p(4).*cosd(3*X(:,3)) + p(5).*cosd(4*X(:,3)) ) ...
  + (X(:,2).^(1/3) ./ X(:,1).^2) .* ( p(6) + p(7).*cosd(X(:,3)) + p(8).*cosd(2*X(:,3)) + p(9).*cosd(3*X(:,3)) + p(10).*cosd(4*X(:,3)) ) ...
  + (1 ./ X(:,1).^3)             .* ( p(11) + p(12).*cosd(X(:,3)) + p(13).*cosd(2*X(:,3)) + p(14).*cosd(3*X(:,3)) + p(15).*cosd(4*X(:,3)) );

X_stat = [df_static.Z, df_static.Sp, df_static.theta];
y_stat = df_static.P_MPa;

p0_stat = 0.01 * ones(1, 15);
p0_stat(1) = 0.065; p0_stat(6) = 0.397; p0_stat(11) = 0.322;

options_stat = optimoptions('lsqcurvefit', 'Algorithm', 'trust-region-reflective', ...
    'Display', 'off', 'MaxFunctionEvaluations', 100000, 'MaxIterations', 5000);

[p_stat, ~] = lsqcurvefit(static_model, p0_stat, X_stat, y_stat, [], [], options_stat);
p_stat = round(p_stat, 4);

A = p_stat(1); B = p_stat(6); C = p_stat(11);

y_stat_calc = static_model(p_stat, X_stat);
R2_stat = 1 - sum((y_stat - y_stat_calc).^2) / sum((y_stat - mean(y_stat)).^2);
MAE_stat = mean(abs(y_stat_calc - y_stat));
MAPE_stat = mean(abs(y_stat_calc - y_stat) ./ y_stat) * 100;

fprintf('\n================ 静爆基准公式 (15参数) ================\n');
fprintf('R² = %.4f | MAE = %.5f MPa | MAPE = %.2f %%\n', R2_stat, MAE_stat, MAPE_stat);
fprintf('========================================================\n');


## 第二步：动爆数据分段 + 全局33参数基准拟合

先拟合一个不分段的 33 参数模型，作为分段拟合的初始猜测值。

In [ ]:
%% 第二步：提取动爆数据，先做不分段33参数拟合作为初始值
disp('>>> 步骤2：全局33参数拟合（用于初始化分段模型）...');

% 提取动爆数据
df_dyn = df(df.V > 0, :);
X_dyn = [df_dyn.Z, df_dyn.Sp, df_dyn.theta, df_dyn.V];
y_dyn = df_dyn.P_MPa;

% 33参数动爆模型 (与 gongshi2.ipynb 完全一致)
% V_n = V/1000 归一化
dyn_model = @(c, X) ...
    (X(:,2).^(2/3) ./ X(:,1)) .* ( c(1) + c(2).*(X(:,4)/1000) + c(3).*(X(:,4)/1000).^2 + ...
        (c(4) + c(5).*(X(:,4)/1000)).*cosd(X(:,3)) + (c(6) + c(7).*(X(:,4)/1000)).*cosd(2*X(:,3)) + ...
        (c(8) + c(9).*(X(:,4)/1000)).*cosd(3*X(:,3)) + (c(10) + c(11).*(X(:,4)/1000)).*cosd(4*X(:,3)) ) ...
    + ...
    (X(:,2).^(1/3) ./ X(:,1).^2) .* ( c(12) + c(13).*(X(:,4)/1000) + c(14).*(X(:,4)/1000).^2 + ...
        (c(15) + c(16).*(X(:,4)/1000)).*cosd(X(:,3)) + (c(17) + c(18).*(X(:,4)/1000)).*cosd(2*X(:,3)) + ...
        (c(19) + c(20).*(X(:,4)/1000)).*cosd(3*X(:,3)) + (c(21) + c(22).*(X(:,4)/1000)).*cosd(4*X(:,3)) ) ...
    + ...
    (1 ./ X(:,1).^3) .* ( c(23) + c(24).*(X(:,4)/1000) + c(25).*(X(:,4)/1000).^2 + ...
        (c(26) + c(27).*(X(:,4)/1000)).*cosd(X(:,3)) + (c(28) + c(29).*(X(:,4)/1000)).*cosd(2*X(:,3)) + ...
        (c(30) + c(31).*(X(:,4)/1000)).*cosd(3*X(:,3)) + (c(32) + c(33).*(X(:,4)/1000)).*cosd(4*X(:,3)) );

% 初始猜测
c0 = 0.05 * ones(1, 33);
c0(1) = A; c0(12) = B; c0(23) = C;

options_dyn = optimoptions('lsqcurvefit', 'Algorithm', 'trust-region-reflective', ...
    'Display', 'off', 'MaxFunctionEvaluations', 500000, 'MaxIterations', 5000);

[c_global, ~] = lsqcurvefit(dyn_model, c0, X_dyn, y_dyn, [], [], options_dyn);
c_global = round(c_global, 4);

% 全局模型误差 (作为对比基线)
P_global = dyn_model(c_global, X_dyn);
R2_global = 1 - sum((y_dyn - P_global).^2) / sum((y_dyn - mean(y_dyn)).^2);
MAPE_global = mean(abs(P_global - y_dyn) ./ y_dyn) * 100;
max_global = max(abs(P_global - y_dyn) ./ y_dyn) * 100;

fprintf('\n【全局33参数基线】\n');
fprintf('  R²=%.4f  MAPE=%.2f%%  最大误差=%.2f%%\n', R2_global, MAPE_global, max_global);


## 第三步：分段拟合

按 Z 值将数据分为两段：
- **段A (Z ≤ 3)**：Z=1, 2, 3 — 中近场
- **段B (Z > 3)**：Z=4 — 远场边界

两段独立拟合同一 33 参数模型结构，初始值均使用全局最优解。

In [ ]:
%% 第三步A：段A拟合 —— Z=1, 2, 3 (中近场)
disp('>>> 步骤3A：段A拟合 (Z=1,2,3，共2940样本)...');

mask_A = df_dyn.Z <= 3;
X_A = X_dyn(mask_A, :);
y_A = y_dyn(mask_A);

fprintf('  段A样本数: %d (Z=1,2,3)\n', sum(mask_A));

% 使用全局最优解作为初始值，信任域反射算法
options_seg = optimoptions('lsqcurvefit', 'Algorithm', 'trust-region-reflective', ...
    'Display', 'iter-detailed', 'MaxFunctionEvaluations', 500000, 'MaxIterations', 10000);

tA = tic;
[c_segA, resnormA, ~, exitflagA] = lsqcurvefit(dyn_model, c_global, X_A, y_A, [], [], options_seg);
tA_elapsed = toc(tA);
c_segA = round(c_segA, 4);

% 段A误差
P_A = dyn_model(c_segA, X_A);
absA = abs(P_A - y_A);
relA = (absA ./ y_A) * 100;
R2_A = 1 - sum((y_A - P_A).^2) / sum((y_A - mean(y_A)).^2);

fprintf('\n【段A (Z=1,2,3) 拟合结果】\n');
fprintf('  耗时=%.1fs  exitflag=%d  残差平方和=%.4f\n', tA_elapsed, exitflagA, resnormA);
fprintf('  R²=%.4f  MAE=%.5f MPa  MAPE=%.2f%%  最大误差=%.2f%%\n', ...
    R2_A, mean(absA), mean(relA), max(relA));


In [ ]:
%% 第三步B：段B拟合 —— Z=4 (远场边界)
disp('>>> 步骤3B：段B拟合 (Z=4，共980样本)...');

mask_B = df_dyn.Z > 3;
X_B = X_dyn(mask_B, :);
y_B = y_dyn(mask_B);

fprintf('  段B样本数: %d (Z=4)\n', sum(mask_B));

% 使用全局最优解作为初始值，但放松约束让优化器有更大自由度
% 因为Z=4的物理行为可能有本质差异
options_segB = optimoptions('lsqcurvefit', 'Algorithm', 'trust-region-reflective', ...
    'Display', 'iter-detailed', 'MaxFunctionEvaluations', 500000, 'MaxIterations', 10000);

tB = tic;
[c_segB, resnormB, ~, exitflagB] = lsqcurvefit(dyn_model, c_global, X_B, y_B, [], [], options_segB);
tB_elapsed = toc(tB);
c_segB = round(c_segB, 4);

% 段B误差
P_B = dyn_model(c_segB, X_B);
absB = abs(P_B - y_B);
relB = (absB ./ y_B) * 100;
R2_B = 1 - sum((y_B - P_B).^2) / sum((y_B - mean(y_B)).^2);

fprintf('\n【段B (Z=4) 拟合结果】\n');
fprintf('  耗时=%.1fs  exitflag=%d  残差平方和=%.4f\n', tB_elapsed, exitflagB, resnormB);
fprintf('  R²=%.4f  MAE=%.5f MPa  MAPE=%.2f%%  最大误差=%.2f%%\n', ...
    R2_B, mean(absB), mean(relB), max(relB));


In [ ]:
%% 第三步C：合并分段预测并计算整体误差
disp('>>> 步骤3C：合并分段结果，计算整体指标...');

% 合并预测值
P_segmented = zeros(size(y_dyn));
P_segmented(mask_A) = P_A;
P_segmented(mask_B) = P_B;

% 存回 DataFrame
df_dyn.P_pred = P_segmented;

% 整体误差
abs_err = abs(P_segmented - y_dyn);
rel_err = (abs_err ./ y_dyn) * 100;
R2_seg = 1 - sum((y_dyn - P_segmented).^2) / sum((y_dyn - mean(y_dyn)).^2);
MAE_seg = mean(abs_err);
MAPE_seg = mean(rel_err);
max_seg = max(rel_err);

fprintf('\n================ 分段模型整体误差 ================\n');
fprintf('R² = %.4f | MAE = %.5f MPa | MAPE = %.2f %% | 最大误差 = %.2f %%\n', ...
    R2_seg, MAE_seg, MAPE_seg, max_seg);
fprintf('\n');

% 分Z段详细统计
fprintf('分Z段详细统计:\n');
fprintf('  Z    | 样本数 | MAPE%%  | 最大%%  | P90%%  | P95%%\n');
fprintf('  -----|--------|--------|--------|--------|-------\n');
for z_val = sort(unique(df_dyn.Z))'
    mask_z = (df_dyn.Z == z_val);
    err_z = rel_err(mask_z);
    fprintf('  Z=%.0f  |  %4d  | %6.2f | %6.2f | %5.2f  | %5.2f\n', ...
        z_val, sum(mask_z), mean(err_z), max(err_z), prctile(err_z, 90), prctile(err_z, 95));
end
fprintf('\n');

% Z=4 分角度排查
fprintf('Z=4 分角度误差 (关键区域):\n');
fprintf('  θ°   | MAPE%%  | 最大%%  | P_pred/P_sim\n');
fprintf('  -----|--------|--------|-----------\n');
for th = sort(unique(df_dyn.theta))'
    mask_th = mask_B & (df_dyn.theta == th);
    if sum(mask_th) > 0
        err_th = rel_err(mask_th);
        ratio_th = mean(P_segmented(mask_th) ./ y_dyn(mask_th));
        fprintf('  θ=%3.0f°| %6.2f | %6.2f | %8.3f\n', th, mean(err_th), max(err_th), ratio_th);
    end
end
fprintf('========================================================\n');


## 第四步：全局 vs 分段 全面对比

In [ ]:
%% 第四步：全局33参数 vs Z分段 全面对比
disp('>>> 步骤4：全局模型 vs 分段模型 对比分析...');

% 全局模型的误差 (重新计算分Z数据)
abs_global = abs(P_global - y_dyn);
rel_global = (abs_global ./ y_dyn) * 100;

fprintf('\n');
fprintf('╔══════════════════════════════════════════════════════════════╗\n');
fprintf('║           全局33参  vs  Z分段2×33参  性能对比                ║\n');
fprintf('╠══════════════════╦═════════════════╦════════════════════════╣\n');
fprintf('║      指标        ║   全局33参      ║   Z分段2×33参        ║\n');
fprintf('╠══════════════════╬═════════════════╬════════════════════════╣\n');
fprintf('║ R²               ║   %.4f        ║   %.4f              ║\n', R2_global, R2_seg);
fprintf('║ MAE (MPa)        ║   %.5f        ║   %.5f              ║\n', mean(abs_global), MAE_seg);
fprintf('║ MAPE (%%)         ║   %.2f         ║   %.2f               ║\n', MAPE_global, MAPE_seg);
fprintf('║ 最大误差 (%%)     ║   %.2f         ║   %.2f               ║\n', max(rel_global), max_seg);
fprintf('║ 90%%分位数 (%%)    ║   %.2f         ║   %.2f               ║\n', ...
    prctile(rel_global, 90), prctile(rel_err, 90));
fprintf('║ 95%%分位数 (%%)    ║   %.2f         ║   %.2f               ║\n', ...
    prctile(rel_global, 95), prctile(rel_err, 95));
fprintf('╚══════════════════╩═════════════════╩════════════════════════╝\n');

% 分Z段最大误差对比
fprintf('\n分Z段最大误差对比:\n');
fprintf('  Z    | 全局最大%%  | 分段最大%%  | 改善量\n');
fprintf('  -----|------------|------------|-------\n');
for z_val = sort(unique(df_dyn.Z))'
    mask_z = (df_dyn.Z == z_val);
    mg = max(rel_global(mask_z));
    ms = max(rel_err(mask_z));
    fprintf('  Z=%.0f  |   %6.2f    |   %6.2f    | %+.2f\n', z_val, mg, ms, mg - ms);
end

% Z=4 分角度对比
fprintf('\nZ=4 各角度最大误差对比 (关键区域):\n');
fprintf('  θ°   | 全局最大%%  | 分段最大%%  | 改善量\n');
fprintf('  -----|------------|------------|-------\n');
for th = sort(unique(df_dyn.theta))'
    mask_th = mask_B & (df_dyn.theta == th);
    if sum(mask_th) > 0
        mg = max(rel_global(mask_th));
        ms = max(rel_err(mask_th));
        fprintf('  θ=%3.0f°|   %6.2f    |   %6.2f    | %+.2f\n', th, mg, ms, mg - ms);
    end
end

fprintf('\n================ 改善总结 ================\n');
fprintf('最大误差:  %.2f%% → %.2f%%  (改善 %.1f 个百分点)\n', max(rel_global), max_seg, max(rel_global) - max_seg);
fprintf('MAPE:     %.2f%% → %.2f%%   (%+.2f 个百分点)\n', MAPE_global, MAPE_seg, MAPE_seg - MAPE_global);
fprintf('R²:       %.4f → %.4f\n', R2_global, R2_seg);
fprintf('===========================================\n');


## 第五步：输出分段公式系数

In [ ]:
%% 第五步：输出两段公式的完整系数
Vn_str = 'V_n = V / 1000';
Sp_str = 'Sp = (T_h / T0)^5^.^2^5^5^8^8';

fprintf('\n');
fprintf('╔══════════════════════════════════════════════════════════════╗\n');
fprintf('║                   Z 分段动爆公式                            ║\n');
fprintf('╠══════════════════════════════════════════════════════════════╣\n');
fprintf('║  使用规则:                                                  ║\n');
fprintf('║    if Z <= 3.5: 使用段A系数 (中近场)                        ║\n');
fprintf('║    if Z >  3.5: 使用段B系数 (远场边界)                      ║\n');
fprintf('╠══════════════════════════════════════════════════════════════╣\n');

fprintf('\n');
fprintf('==================== 段A: Z=1,2,3 (中近场) ====================\n');
fprintf('P_dyn_A = Term1 + Term2 + Term3\n');
fprintf('\n');
fprintf('【Term1】= (Sp^(2/3)/Z) * [\n');
fprintf('  常数:  %+.4f %+.4f*Vn %+.4f*Vn²\n', c_segA(1:3));
fprintf('  cosθ:  %+.4f %+.4f*Vn\n', c_segA(4:5));
fprintf('  cos2θ: %+.4f %+.4f*Vn\n', c_segA(6:7));
fprintf('  cos3θ: %+.4f %+.4f*Vn\n', c_segA(8:9));
fprintf('  cos4θ: %+.4f %+.4f*Vn ]\n', c_segA(10:11));
fprintf('\n');
fprintf('【Term2】= (Sp^(1/3)/Z²) * [\n');
fprintf('  常数:  %+.4f %+.4f*Vn %+.4f*Vn²\n', c_segA(12:14));
fprintf('  cosθ:  %+.4f %+.4f*Vn\n', c_segA(15:16));
fprintf('  cos2θ: %+.4f %+.4f*Vn\n', c_segA(17:18));
fprintf('  cos3θ: %+.4f %+.4f*Vn\n', c_segA(19:20));
fprintf('  cos4θ: %+.4f %+.4f*Vn ]\n', c_segA(21:22));
fprintf('\n');
fprintf('【Term3】= (1/Z³) * [\n');
fprintf('  常数:  %+.4f %+.4f*Vn %+.4f*Vn²\n', c_segA(23:25));
fprintf('  cosθ:  %+.4f %+.4f*Vn\n', c_segA(26:27));
fprintf('  cos2θ: %+.4f %+.4f*Vn\n', c_segA(28:29));
fprintf('  cos3θ: %+.4f %+.4f*Vn\n', c_segA(30:31));
fprintf('  cos4θ: %+.4f %+.4f*Vn ]\n', c_segA(32:33));
fprintf('=============================================================\n');

fprintf('\n');
fprintf('==================== 段B: Z=4 (远场边界) ====================\n');
fprintf('P_dyn_B = Term1 + Term2 + Term3\n');
fprintf('\n');
fprintf('【Term1】= (Sp^(2/3)/Z) * [\n');
fprintf('  常数:  %+.4f %+.4f*Vn %+.4f*Vn²\n', c_segB(1:3));
fprintf('  cosθ:  %+.4f %+.4f*Vn\n', c_segB(4:5));
fprintf('  cos2θ: %+.4f %+.4f*Vn\n', c_segB(6:7));
fprintf('  cos3θ: %+.4f %+.4f*Vn\n', c_segB(8:9));
fprintf('  cos4θ: %+.4f %+.4f*Vn ]\n', c_segB(10:11));
fprintf('\n');
fprintf('【Term2】= (Sp^(1/3)/Z²) * [\n');
fprintf('  常数:  %+.4f %+.4f*Vn %+.4f*Vn²\n', c_segB(12:14));
fprintf('  cosθ:  %+.4f %+.4f*Vn\n', c_segB(15:16));
fprintf('  cos2θ: %+.4f %+.4f*Vn\n', c_segB(17:18));
fprintf('  cos3θ: %+.4f %+.4f*Vn\n', c_segB(19:20));
fprintf('  cos4θ: %+.4f %+.4f*Vn ]\n', c_segB(21:22));
fprintf('\n');
fprintf('【Term3】= (1/Z³) * [\n');
fprintf('  常数:  %+.4f %+.4f*Vn %+.4f*Vn²\n', c_segB(23:25));
fprintf('  cosθ:  %+.4f %+.4f*Vn\n', c_segB(26:27));
fprintf('  cos2θ: %+.4f %+.4f*Vn\n', c_segB(28:29));
fprintf('  cos3θ: %+.4f %+.4f*Vn\n', c_segB(30:31));
fprintf('  cos4θ: %+.4f %+.4f*Vn ]\n', c_segB(32:33));
fprintf('=============================================================\n');


## 第六步：导出误差分析 Excel

In [ ]:
%% 第六步：导出 Excel 误差分析表
disp('>>> 步骤6：导出误差分析Excel文件...');

% ==================== 动爆误差表 ====================
df_out = df_dyn(:, {'M', 'H', 'V', 'Z', 'theta', 'P_MPa'});
df_out.P_pred = P_segmented;
df_out.Abs_Error = abs_err;
df_out.Rel_Error_Pct = rel_err;
df_out.Error_Magnitude = abs(rel_err);
df_out.Segment = repmat({'SegA_Z123'}, height(df_dyn), 1);
df_out.Segment(mask_B) = {'SegB_Z4'};

% 中文表头
df_out.Properties.VariableNames{'P_pred'} = 'P_拟合计算值_MPa';
df_out.Properties.VariableNames{'Abs_Error'} = '绝对误差_MPa';
df_out.Properties.VariableNames{'Rel_Error_Pct'} = '相对误差_百分比';
df_out.Properties.VariableNames{'Error_Magnitude'} = '误差绝对大小_用于排序';
df_out.Properties.VariableNames{'Segment'} = '所属分段';

% 版本A：按工况排序
df_by_condition = sortrows(df_out, {'H', 'V', 'Z', 'theta'});
filename_A = 'Z分段_优化结果_按工况排序.xlsx';
writetable(df_by_condition, filename_A);
fprintf('✅ 表格A: 【%s】\n', filename_A);

% 版本B：按误差大小降序
df_by_error = sortrows(df_out, '误差绝对大小_用于排序', 'descend');
filename_B = 'Z分段_优化结果_按误差大小排序.xlsx';
writetable(df_by_error, filename_B);
fprintf('✅ 表格B: 【%s】\n', filename_B);

% ==================== 静爆误差表 ====================
df_static_out = df_static(:, {'M', 'H', 'Z', 'theta', 'P_MPa'});
y_stat_pred = static_model(p_stat, [df_static.Z, df_static.Sp, df_static.theta]);
df_static_out.P_pred = y_stat_pred;
df_static_out.Abs_Error = abs(y_stat_pred - df_static.P_MPa);
df_static_out.Rel_Error_Pct = (df_static_out.Abs_Error ./ df_static.P_MPa) * 100;
df_static_out.Error_Magnitude = abs(df_static_out.Rel_Error_Pct);

df_static_out.Properties.VariableNames{'P_pred'} = 'P_拟合计算值_MPa';
df_static_out.Properties.VariableNames{'Abs_Error'} = '绝对误差_MPa';
df_static_out.Properties.VariableNames{'Rel_Error_Pct'} = '相对误差_百分比';
df_static_out.Properties.VariableNames{'Error_Magnitude'} = '误差绝对大小_用于排序';

df_static_sorted = sortrows(df_static_out, '误差绝对大小_用于排序', 'descend');
filename_S = 'Z分段_优化结果_静爆误差表.xlsx';
writetable(df_static_sorted, filename_S);
fprintf('✅ 表格C: 【%s】\n', filename_S);

fprintf('\n🎉 Z分段拟合全部完成！所有结果已导出至当前文件夹。\n');
